# Netflix Churn Prediction — Featuretools Demo
This notebook demonstrates automated feature engineering using Featuretools 
on the Netflix User Behavior dataset.

## 1. Import Libraries
We use `pandas` for data manipulation, `featuretools` for automated feature engineering, 
and `pathlib` for file path management.

In [7]:
import pandas as pd
from pathlib import Path
import featuretools as ft

## 2. Load Data
Loading 5 pre-cleaned CSV files from the Netflix User Behavior dataset:
- **users** — demographic and subscription info per user
- **watch** — watch history and duration
- **recs** — recommendation interactions
- **search** — search queries and behavior
- **reviews** — user ratings and sentiment## 2. Load Data

In [12]:
#load cleaned data

BASE_DIR = Path("/Users/urvashijha/Documents/MSDS/DATA515/project/Netflix-User-Behavior")

DATA_DIR = BASE_DIR / "Cleaned_data"

users   = pd.read_csv(DATA_DIR / "users_cleaned.csv")
watch   = pd.read_csv(DATA_DIR / "watch_cleaned.csv")
recs    = pd.read_csv(DATA_DIR / "recs_cleaned.csv")
search  = pd.read_csv(DATA_DIR / "search_cleaned.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_cleaned.csv")

# Combine all dataframes into a dictionary for easy access
dataframes = {
    "users":   users,
    "watch":   watch,
    "recs":    recs,
    "search":  search,
    "reviews": reviews,
}

## 3. Featuretools Demo
This cell:
1. Samples 200 users from the dataset
2. Cleans and prepares all 5 tables
3. Builds a Featuretools EntitySet connecting tables via `user_id`
4. Runs Deep Feature Synthesis to generate 46 features automatically


In [ ]:
# featauretools demo for feature engineering 

SAMPLE_USERS = 200  # small number for demo to prevent feature explosion

# -----------------------------
# 1. Sample users
# -----------------------------
sample_user_ids = users["user_id"].sample(SAMPLE_USERS, random_state=42)

users_s   = users[users["user_id"].isin(sample_user_ids)].copy()
watch_s   = watch[watch["user_id"].isin(sample_user_ids)].copy()
recs_s    = recs[recs["user_id"].isin(sample_user_ids)].copy()
search_s  = search[search["user_id"].isin(sample_user_ids)].copy()
reviews_s = reviews[reviews["user_id"].isin(sample_user_ids)].copy()

# -----------------------------
# 2. Minimal cleanup
# -----------------------------
users_s = users_s.drop(columns=["email", "first_name", "last_name"], errors="ignore")
reviews_s = reviews_s.drop(columns=["review_text"], errors="ignore")

# Convert time columns
watch_s["watch_date"]            = pd.to_datetime(watch_s["watch_date"], format = "mixed")
recs_s["recommendation_date"]   = pd.to_datetime(recs_s["recommendation_date"], format = "mixed")
search_s["search_date"]          = pd.to_datetime(search_s["search_date"], format = "mixed")
reviews_s["review_date"]         = pd.to_datetime(reviews_s["review_date"], format = "mixed")

# Create simple unique IDs
watch_s["watch_id"]    = range(len(watch_s))
recs_s["rec_id"]       = range(len(recs_s))
search_s["search_id"]  = range(len(search_s))
reviews_s["review_id"] = range(len(reviews_s))

# Ensure column names are strings
for df in [users_s, watch_s, recs_s, search_s, reviews_s]:
    df.columns = df.columns.astype(str)

# -----------------------------
# 3. Build EntitySet
# -----------------------------
es = ft.EntitySet(id="netflix_demo")

es = es.add_dataframe(dataframe_name="users",   dataframe=users_s,   index="user_id")
es = es.add_dataframe(dataframe_name="watch",   dataframe=watch_s,   index="watch_id",   time_index="watch_date")
es = es.add_dataframe(dataframe_name="recs",    dataframe=recs_s,    index="rec_id",     time_index="recommendation_date")
es = es.add_dataframe(dataframe_name="search",  dataframe=search_s,  index="search_id",  time_index="search_date")
es = es.add_dataframe(dataframe_name="reviews", dataframe=reviews_s, index="review_id",  time_index="review_date")

# Relationships
es = es.add_relationship("users", "user_id", "watch",   "user_id")
es = es.add_relationship("users", "user_id", "recs",    "user_id")
es = es.add_relationship("users", "user_id", "search",  "user_id")
es = es.add_relationship("users", "user_id", "reviews", "user_id")

print(es)

# -----------------------------
# 4. Run DFS
# -----------------------------
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="users",
    agg_primitives=["count", "mean", "num_unique"],
    max_depth=1,
    verbose=True,
)



In [26]:
# Print the results
print("Features generated:", len(feature_defs))
print("Feature matrix shape:", feature_matrix.shape)
print(feature_matrix.head())


# save it as csv
feature_matrix.to_csv(BASE_DIR / "feature_matrix.csv")

Features generated: 46
Feature matrix shape: (200, 46)
             age             gender country state_province subscription_plan  \
user_id                                                                        
user_00026  60.0  Prefer not to say  Canada        Ontario             Basic   
user_00045  40.0             Female     USA   Pennsylvania           Premium   
user_00088  38.0              Other  Canada        Alberta          Standard   
user_00135  16.0               Male     USA        Georgia           Premium   
user_00184  19.0             Female     USA        Indiana           Premium   

            is_active  monthly_spend  primary_device  household_size  \
user_id                                                                
user_00026       True          36.53        Smart TV             2.0   
user_00045       True           3.22         Desktop             4.0   
user_00088       True          10.87          Laptop             4.0   
user_00135       True   